In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files into DataFrames (keep full DataFrames in memory for later steps)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to add timezone-aware datetime column (UTC -> Asia/Shanghai)
def add_dt_col(df, ts_col='timestamp', col_name='dt'):
    # Ensure integer timestamp
    df[ts_col] = pd.to_numeric(df[ts_col], errors='coerce').astype('Int64')
    # convert to tz-aware datetime
    df[col_name] = pd.to_datetime(df[ts_col].astype('Int64'), unit='s', errors='coerce').dt.tz_localize('UTC').dt.tz_convert(tz)
    return df

df_metric = add_dt_col(df_metric)
df_trace = add_dt_col(df_trace)
df_log = add_dt_col(df_log)
df_error = add_dt_col(df_error)

# Summarize function
def compact_summary(df, name_col=None, sample_cols=None):
    total_rows = int(df.shape[0])
    if df['dt'].notna().any():
        min_ts = df['dt'].min()
        max_ts = df['dt'].max()
        # represent as ISO strings
        min_ts_s = min_ts.isoformat()
        max_ts_s = max_ts.isoformat()
    else:
        min_ts_s = None
        max_ts_s = None
    unique_cmdb = list(pd.Series(df['cmdb_id'].astype(str).fillna('')).unique())[:20]
    unique_names = None
    if name_col and name_col in df.columns:
        unique_names = list(pd.Series(df[name_col].astype(str).fillna('')).unique())[:20]
    # sample: select relevant columns to keep output compact
    if sample_cols is None:
        sample_cols = ['timestamp', 'dt', 'cmdb_id']
        if name_col and name_col in df.columns:
            sample_cols.append(name_col)
        # include value/message if present
        for c in ['value', 'message']:
            if c in df.columns and c not in sample_cols:
                sample_cols.append(c)
    sample = df.loc[:, [c for c in sample_cols if c in df.columns]].head(5)
    return {
        'total_rows': total_rows,
        'min_timestamp': min_ts_s,
        'max_timestamp': max_ts_s,
        'unique_cmdb_id_top20': unique_cmdb,
        'unique_names_top20': unique_names,
        'sample_head_5': sample
    }

summary_metric = compact_summary(df_metric, name_col='kpi_name', sample_cols=['timestamp','dt','cmdb_id','kpi_name','value'])
summary_trace = compact_summary(df_trace, name_col='trace_name', sample_cols=['timestamp','dt','cmdb_id','trace_name','value'])
summary_log = compact_summary(df_log, name_col='log_name', sample_cols=['timestamp','dt','cmdb_id','log_name','value'])
summary_error = compact_summary(df_error, name_col='message', sample_cols=['timestamp','dt','cmdb_id','message'])

summaries = {
    'metric.csv': summary_metric,
    'trace.csv': summary_trace,
    'log.csv': summary_log,
    'error_logs.csv': summary_error
}

summaries

```
Out[1]:
```
```python
# Summarize loaded telemetry files in plain English (compact)
# Uses existing DataFrame variables: df_metric, df_trace, df_log, df_error

def fmt_range(df):
    if 'dt' in df.columns and df['dt'].notna().any():
        mn = df['dt'].min().isoformat()
        mx = df['dt'].max().isoformat()
        return f"{mn} to {mx} (UTC+8)"
    return "no timestamps"

def top_list(series, n=10):
    vals = list(series.dropna().astype(str).unique())[:n]
    return ", ".join(vals) if vals else "[]"

summary = (
    "Telemetry files summary (plain English):\n\n"
    f"metric.csv: {df_metric.shape[0]} rows; time range {fmt_range(df_metric)}.\n"
    f"  Unique cmdb_id examples (up to 20): {top_list(df_metric['cmdb_id'], 20)}.\n"
    f"  Unique KPI names: showing first 5 examples: {top_list(df_metric['kpi_name'],5)}; total distinct KPI names: {df_metric['kpi_name'].nunique()}.\n\n"
    f"trace.csv: {df_trace.shape[0]} rows; time range {fmt_range(df_trace)}.\n"
    f"  Unique cmdb_id examples (up to 20): {top_list(df_trace['cmdb_id'], 20)}.\n"
    f"  Unique trace names: showing first 5 examples: {top_list(df_trace['trace_name'],5)}; total distinct trace names: {df_trace['trace_name'].nunique()}.\n\n"
    f"log.csv: {df_log.shape[0]} rows; time range {fmt_range(df_log)}.\n"
    f"  Unique cmdb_id examples (up to 20): {top_list(df_log['cmdb_id'], 20)}.\n"
    f"  Unique log names: {top_list(df_log['log_name'],5)}; total distinct log names: {df_log['log_name'].nunique()}.\n\n"
    f"error_logs.csv: {df_error.shape[0]} rows; {fmt_range(df_error)}.\n"
    "  Note: error_logs.csv contains zero rows (no messages present).\n\n"
    "Samples: first 5 rows of each file are kept in memory (not printed here)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 28120, 'min_timestamp': '2021-03-04T01:00:00+08:00', 'max_timestamp': '2021-03-04T01:30:00+08:00', 'unique_cmdb_id_top20': ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02'], 'unique_names_top20': ['JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'JVM-Operating System_7778_JVM_JVM_CPULoad', 'JVM-Threads_7778_JVM_ThreadCount_Threads', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_FileSizeMB', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvgServ', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvgServ', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps'], 'sample_head_5':     timestamp                        dt cmdb_id                                     kpi_name         value
0  1614790800 2021-03-04 01:00:00+08:00    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1  1614790800 2021-03-04 01:00:00+08:00    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  2.942630e+01
2  1614790800 2021-03-04 01:00:00+08:00    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  6.136996e+08
3  1614790800 2021-03-04 01:00:00+08:00    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  8.265500e+08
4  1614790800 2021-03-04 01:00:00+08:00    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad  7.460000e-02}, 'trace.csv': {'total_rows': 6327, 'min_timestamp': '2021-03-04T01:00:00+08:00', 'max_timestamp': '2021-03-04T01:29:00+08:00', 'unique_cmdb_id_top20': ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 'unique_names_top20': ['trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_Tomcat01.duration_mean', 'trace.to_Tomcat01.duration_p95', 'trace.to_Tomcat01.row_count', 'trace.to_Tomcat02.duration_mean', 'trace.to_Tomcat02.duration_p95', 'trace.to_Tomcat02.row_count', 'trace.to_Tomcat03.duration_mean', 'trace.to_Tomcat03.duration_p95', 'trace.to_Tomcat03.row_count', 'trace.to_Tomcat04.duration_mean', 'trace.to_Tomcat04.duration_p95', 'trace.to_Tomcat04.row_count', 'trace.from_Tomcat01.duration_mean', 'trace.from_Tomcat01.duration_p95', 'trace.from_Tomcat01.row_count', 'trace.from_Tomcat02.duration_mean', 'trace.from_Tomcat02.duration_p95'], 'sample_head_5':     timestamp                        dt cmdb_id                       trace_name      value
0  1614790800 2021-03-04 01:00:00+08:00    IG01         trace.self.duration_mean   0.260793
1  1614790800 2021-03-04 01:00:00+08:00    IG01          trace.self.duration_p95   0.581000
2  1614790800 2021-03-04 01:00:00+08:00    IG01             trace.self.row_count  58.000000
3  1614790800 2021-03-04 01:00:00+08:00    IG01  trace.to_Tomcat01.duration_mean   0.239714
4  1614790800 2021-03-04 01:00:00+08:00    IG01   trace.to_Tomcat01.duration_p95   0.513900}, 'log.csv': {'total_rows': 432, 'min_timestamp': '2021-03-04T01:00:00+08:00', 'max_timestamp': '2021-03-04T01:30:00+08:00', 'unique_cmdb_id_top20': ['Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02'], 'unique_names_top20': ['log.error_count', 'log.gc_pause_max', 'log.row_count'], 'sample_head_5':     timestamp                        dt   cmdb_id          log_name  value
0  1614790800 2021-03-04 01:00:00+08:00  Tomcat01   log.error_count   0.00
1  1614790800 2021-03-04 01:00:00+08:00  Tomcat01  log.gc_pause_max   5.12
2  1614790800 2021-03-04 01:00:00+08:00  Tomcat01     log.row_count  87.00
3  1614790800 2021-03-04 01:00:00+08:00  Tomcat02   log.error_count   0.00
4  1614790800 2021-03-04 01:00:00+08:00  Tomcat02  log.gc_pause_max   5.12}, 'error_logs.csv': {'total_rows': 0, 'min_timestamp': None, 'max_timestamp': None, 'unique_cmdb_id_top20': [], 'unique_names_top20': [], 'sample_head_5': Empty DataFrame
Columns: [timestamp, dt, cmdb_id, message]
Index: []}}```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing df_metric from kernel
# Ensure value is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using all data (before filtering)
df_global = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    global_p95=('value', lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan),
    global_p05=('value', lambda x: np.nanpercentile(x.dropna(), 5) if x.dropna().size>0 else np.nan)
)

# Count distinct pairs in full metric.csv
distinct_pairs = df_global.shape[0]

# 2) Filter df_metric to the incident window [1614790800, 1614792600] inclusive
start_ts = 1614790800
end_ts = 1614792600
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Merge global thresholds into windowed data and compute anomalies
df_win_thresh = df_window.merge(df_global, on=['cmdb_id', 'kpi_name'], how='left')

# Flags for anomalies (>= P95 and <= P05)
df_win_thresh['is_high'] = df_win_thresh['value'] >= df_win_thresh['global_p95']
df_win_thresh['is_low'] = df_win_thresh['value'] <= df_win_thresh['global_p05']

# Total points per pair in window
total_counts = df_win_thresh.groupby(['cmdb_id', 'kpi_name'], as_index=False).size().rename(columns={'size':'total_points_window'})

# High anomaly stats
high_grp = df_win_thresh[df_win_thresh['is_high']].groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    high_anomaly_count=('is_high','sum'),
    earliest_high_anomaly_dt=('dt','min')
)

# Low anomaly stats
low_grp = df_win_thresh[df_win_thresh['is_low']].groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    low_anomaly_count=('is_low','sum'),
    earliest_low_anomaly_dt=('dt','min')
)

# 4) Assemble result table by merging
df_result = df_global.merge(total_counts, on=['cmdb_id','kpi_name'], how='left')
df_result = df_result.merge(high_grp, on=['cmdb_id','kpi_name'], how='left')
df_result = df_result.merge(low_grp, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0 and keep datetime NaT as None
df_result['total_points_window'] = df_result['total_points_window'].fillna(0).astype(int)
df_result['high_anomaly_count'] = df_result['high_anomaly_count'].fillna(0).astype(int)
df_result['low_anomaly_count'] = df_result['low_anomaly_count'].fillna(0).astype(int)

# Convert earliest dt columns to ISO strings in UTC+8, or None
def dt_to_iso(s):
    return s.dt.tz_convert(pytz.timezone('Asia/Shanghai')).dt.strftime('%Y-%m-%dT%H:%M:%S%z') if isinstance(s.dtype, pd.DatetimeTZDtype) or pd.api.types.is_datetime64tz_dtype(s) else s

# Apply conversion carefully (handle NaT)
if 'earliest_high_anomaly_dt' in df_result.columns:
    df_result['earliest_high_anomaly_timestamp'] = df_result['earliest_high_anomaly_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
else:
    df_result['earliest_high_anomaly_timestamp'] = None

if 'earliest_low_anomaly_dt' in df_result.columns:
    df_result['earliest_low_anomaly_timestamp'] = df_result['earliest_low_anomaly_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
else:
    df_result['earliest_low_anomaly_timestamp'] = None

# Compute combined anomaly count for sorting
df_result['combined_anomaly_count'] = df_result['high_anomaly_count'] + df_result['low_anomaly_count']

# Select columns and sort by combined anomaly count desc, then total_points_window desc
out_cols = [
    'cmdb_id', 'kpi_name', 'global_p95', 'global_p05',
    'total_points_window',
    'high_anomaly_count', 'earliest_high_anomaly_timestamp',
    'low_anomaly_count', 'earliest_low_anomaly_timestamp',
    'combined_anomaly_count'
]
df_result_final = df_result.loc[:, out_cols].sort_values(['combined_anomaly_count','total_points_window'], ascending=[False, False])

# Limit to top 20 rows for compactness
df_result_top20 = df_result_final.head(20)

# Keep intermediate tables in memory: df_global, df_window, df_win_thresh, total_counts, high_grp, low_grp
df_global.shape, df_window.shape, df_result_top20

```
Out[2]:
```
# Plain-English summary of the computed results (compact)
# Reuse existing kernel variables: df_global, df_window, df_result_top20, distinct_pairs (if present)

distinct = int(df_global.shape[0]) if 'df_global' in globals() else ('unknown')
rows_window = int(df_window.shape[0]) if 'df_window' in globals() else ('unknown')

# Prepare brief top examples (up to 10) from df_result_top20
top_examples = []
if 'df_result_top20' in globals():
    display_rows = df_result_top20.head(10)
    for _, r in display_rows.iterrows():
        top_examples.append(
            f"{r['cmdb_id']} | {r['kpi_name']} | p95={r['global_p95']:.6g} p05={r['global_p05']:.6g} "
            f"points={int(r['total_points_window'])} high={int(r['high_anomaly_count'])} earliest_high={r['earliest_high_anomaly_timestamp']} "
            f"low={int(r['low_anomaly_count'])} earliest_low={r['earliest_low_anomaly_timestamp']} combined={int(r['combined_anomaly_count'])}"
        )

summary = (
    "Summary of anomaly analysis (plain English):\n\n"
    f"- Total distinct (cmdb_id, kpi_name) pairs in metric.csv: {distinct}.\n"
    f"- Incident window: 2021-03-04 01:00:00+08 to 2021-03-04 01:30:00+08 "
    f"(timestamps 1614790800 to 1614792600). Rows in this window: {rows_window}.\n\n"
    "- What was computed:\n"
    "  1) Global P95 and P05 were computed per (cmdb_id, kpi_name) using all available metric data.\n"
    "  2) The metric data was filtered to the incident window above.\n"
    "  3) For each pair in the window we counted total points, high anomalies (value >= global P95) and low anomalies (value <= global P05), and recorded earliest timestamps of each anomaly type.\n\n"
    "- Key observation from the top results:\n"
    "  Many top-ranked (cmdb_id, kpi_name) pairs (examples below) show every point in the window flagged as both high and low (i.e., high_count == low_count == 31 and combined == 62). "
    "This happens when global_p95 == global_p05 == the observed value, so every sample meets both thresholds.\n\n"
    "Top example pairs (up to 10):\n"
    + ("\n".join(f"  - {line}" for line in top_examples) if top_examples else "  (no top rows available)")
    + "\n\n- Intermediate tables retained in memory for follow-up: df_global, df_window, df_win_thresh, total_counts, high_grp, low_grp, df_result_top20.\n"
    "- If you want, I can now (a) inspect why many metrics have identical P95/P05, (b) recompute thresholds with a different method, or (c) produce a concise list of pairs with non-zero anomalies only."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

((1686, 4), (28120, 5),      cmdb_id                                           kpi_name    global_p95    global_p05  total_points_window  high_anomaly_count earliest_high_anomaly_timestamp  low_anomaly_count earliest_low_anomaly_timestamp  combined_anomaly_count
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  8.265500e+08  8.265500e+08                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
66      IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
67      IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
77      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00  1.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
80      IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01  3.000000e-01                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
81      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00  3.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
88      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
154     IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
155     IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
165     IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00  1.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
169     IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00  3.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
176     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
179     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.060958e+08  5.060958e+08                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
269     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
571  Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
572  Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
574  Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  0.000000e+00  0.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
578  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  7.000000e+00  7.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
793  Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00                   31                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62)```
```

In [ ]:
# In[3]:

# Reuse in-memory tables: df_global, df_window, df_win_thresh (if exists); otherwise rebuild df_win_thresh
import pandas as pd
import pytz
allowed = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04","MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Rebuild df_win_thresh if not present
if 'df_win_thresh' not in globals():
    df_win_thresh = df_window.merge(df_global, on=['cmdb_id','kpi_name'], how='left')
    df_win_thresh['is_high'] = df_win_thresh['value'] >= df_win_thresh['global_p95']
    df_win_thresh['is_low'] = df_win_thresh['value'] <= df_win_thresh['global_p05']

# Aggregate counts and earliest anomaly times within the window
total_counts = df_win_thresh.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    total_points_window=('value','size')
)

high_grp = df_win_thresh[df_win_thresh['is_high']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    high_anomaly_count=('is_high','sum'),
    earliest_high_dt=('dt','min')
)

low_grp = df_win_thresh[df_win_thresh['is_low']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    low_anomaly_count=('is_low','sum'),
    earliest_low_dt=('dt','min')
)

# Assemble full result by merging global thresholds with window stats
df_candidates = df_global.merge(total_counts, on=['cmdb_id','kpi_name'], how='left')
df_candidates = df_candidates.merge(high_grp, on=['cmdb_id','kpi_name'], how='left')
df_candidates = df_candidates.merge(low_grp, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0
df_candidates['total_points_window'] = df_candidates['total_points_window'].fillna(0).astype(int)
df_candidates['high_anomaly_count'] = df_candidates['high_anomaly_count'].fillna(0).astype(int)
df_candidates['low_anomaly_count'] = df_candidates['low_anomaly_count'].fillna(0).astype(int)

# Convert earliest dt to ISO strings (UTC+8). dt values are timezone-aware already.
df_candidates['earliest_high_anomaly_timestamp'] = df_candidates['earliest_high_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
df_candidates['earliest_low_anomaly_timestamp'] = df_candidates['earliest_low_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Combined anomaly count
df_candidates['combined_anomaly_count'] = df_candidates['high_anomaly_count'] + df_candidates['low_anomaly_count']

# Apply filters:
# 1) only allowed cmdb_id
# 2) exclude rows where global_p95 == global_p05 (handle NaN safely)
# 3) include only rows with combined_anomaly_count > 0
mask_allowed = df_candidates['cmdb_id'].isin(allowed)
mask_not_constant = ~(df_candidates['global_p95'].notna() & df_candidates['global_p05'].notna() & (df_candidates['global_p95'] == df_candidates['global_p05']))
mask_combined = df_candidates['combined_anomaly_count'] > 0

df_candidates_filtered = df_candidates[mask_allowed & mask_not_constant & mask_combined].copy()

# Sort and select top 20
df_candidates_filtered = df_candidates_filtered.sort_values(['combined_anomaly_count','total_points_window'], ascending=[False, False])

out_cols = [
    'cmdb_id', 'kpi_name', 'global_p95', 'global_p05',
    'total_points_window',
    'high_anomaly_count', 'earliest_high_anomaly_timestamp',
    'low_anomaly_count', 'earliest_low_anomaly_timestamp',
    'combined_anomaly_count'
]

df_candidates_top20 = df_candidates_filtered.loc[:, out_cols].head(20)
candidates_count = int(df_candidates_filtered.shape[0])

# Display compact results
df_candidates_top20, candidates_count

```
Out[3]:
```
```python
# Plain-English summary of candidate root-cause components (compact)
# Reuse in-memory tables: df_candidates_top20, candidates_count

top_lines = []
if 'df_candidates_top20' in globals():
    for _, r in df_candidates_top20.head(10).iterrows():
        top_lines.append(
            f"{r['cmdb_id']} | {r['kpi_name']} | p95={r['global_p95']:.6g} p05={r['global_p05']:.6g} "
            f"points={int(r['total_points_window'])} high={int(r['high_anomaly_count'])} earliest_high={r['earliest_high_anomaly_timestamp']} "
            f"low={int(r['low_anomaly_count'])} earliest_low={r['earliest_low_anomaly_timestamp']} combined={int(r['combined_anomaly_count'])}"
        )

summary = (
    "Candidate components summary (plain English):\n\n"
    "- Filtering applied: kept only allowed components, excluded constant KPIs where global_p95 == global_p05, "
    "and required combined_anomaly_count > 0.\n\n"
    f"- Total candidate (cmdb_id, kpi_name) pairs after filtering: {int(candidates_count)}.\n\n"
    "- Top candidate examples (up to 10):\n"
    + ("\n".join("  - " + line for line in top_lines) if top_lines else "  (no candidates to show)") 
    + "\n\n"
    "- Notes: Each example shows component | KPI name | global P95 / P05 | points in window | high count & earliest time | low count & earliest time | combined count.\n"
    "- If you want, I can (a) expand the list to all candidates, (b) rank by other criteria, or (c) produce a short prioritized remediation checklist for these components."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name    global_p95    global_p05  total_points_window  high_anomaly_count earliest_high_anomaly_timestamp  low_anomaly_count earliest_low_anomaly_timestamp  combined_anomaly_count
91        IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  8.242818e+08  8.242816e+08                   31                   4       2021-03-04T01:27:00+08:00                 27      2021-03-04T01:00:00+08:00                      31
168       IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.500000e-01  3.000000e-01                   31                   2       2021-03-04T01:15:00+08:00                 29      2021-03-04T01:00:00+08:00                      31
272       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  6.889296e+08  6.889212e+08                   31                   5       2021-03-04T01:26:00+08:00                 26      2021-03-04T01:00:00+08:00                      31
577    Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  4.000000e-01  3.000000e-01                   31                  27       2021-03-04T01:00:00+08:00                  4      2021-03-04T01:01:00+08:00                      31
799    Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  5.000000e-01  4.000000e-01                   31                   3       2021-03-04T01:03:00+08:00                 28      2021-03-04T01:00:00+08:00                      31
881    Redis01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  4.000000e-01  3.000000e-01                   31                  14       2021-03-04T01:03:00+08:00                 17      2021-03-04T01:00:00+08:00                      31
894    Redis01        redis-Redis_6379_Redis  (connected_clients)  6.010000e+02  5.990000e+02                   31                  28       2021-03-04T01:00:00+08:00                  3      2021-03-04T01:05:00+08:00                      31
1011   Redis02          redis-Redis_6379_Redis  (keyspace_misses)  1.989000e+04  1.988200e+04                   31                   6       2021-03-04T01:25:00+08:00                 25      2021-03-04T01:00:00+08:00                      31
1181  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  5.000000e-01  0.000000e+00                   31                   2       2021-03-04T01:09:00+08:00                 29      2021-03-04T01:00:00+08:00                      31
1290  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+00  0.000000e+00                   31                   3       2021-03-04T01:07:00+08:00                 28      2021-03-04T01:00:00+08:00                      31
1582  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  5.000000e-01  4.000000e-01                   31                  12       2021-03-04T01:00:00+08:00                 19      2021-03-04T01:01:00+08:00                      31
1660  apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  4.000000e-01  3.000000e-01                   31                   6       2021-03-04T01:01:00+08:00                 25      2021-03-04T01:00:00+08:00                      31
1016   Redis02  redis-Redis_6379_Redis  (mem_fragmentation_ratio)  5.700000e-01  5.600000e-01                   30                  23       2021-03-04T01:01:00+08:00                  7      2021-03-04T01:09:00+08:00                      30
566    Mysql01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem  4.740000e+02  4.730000e+02                   23                  13       2021-03-04T01:13:00+08:00                 10      2021-03-04T01:00:00+08:00                      23
151       IG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.000000e+01  8.900000e+01                   22                  10       2021-03-04T01:16:00+08:00                 12      2021-03-04T01:00:00+08:00                      22
1650  apache02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  4.695000e+01  4.600000e+01                   22                   2       2021-03-04T01:13:00+08:00                 20      2021-03-04T01:00:00+08:00                      22
100       IG02                            OSLinux-CPU_CPU_CPULoad  1.080000e+00  1.000000e+00                   31                   3       2021-03-04T01:07:00+08:00                 18      2021-03-04T01:00:00+08:00                      21
244       MG01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  6.900000e+01  6.800000e+01                   21                  19       2021-03-04T01:03:00+08:00                  2      2021-03-04T01:00:00+08:00                      21
338       MG02       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  4.931310e+01  4.930030e+01                   21                  10       2021-03-04T01:05:00+08:00                 11      2021-03-04T01:01:00+08:00                      21
1137  Tomcat01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.200000e+01  9.100000e+01                   21                  18       2021-03-04T01:00:00+08:00                  3      2021-03-04T01:01:00+08:00                      21, 782)```
```

In [ ]:
# In[4]:

# Reuse existing in-memory variables where possible: df_global, df_window
import pandas as pd
import numpy as np
import pytz

allowed = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
           "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Rebuild windowed merged table (thresholds computed globally in df_global)
df_win2 = df_window.merge(df_global, on=['cmdb_id','kpi_name'], how='left')

# Ensure numeric and dt present
df_win2['value'] = pd.to_numeric(df_win2['value'], errors='coerce')
# is_high / is_low using global thresholds (global_p95/global_p05 computed from full series)
df_win2['is_high'] = df_win2['value'] >= df_win2['global_p95']
df_win2['is_low']  = df_win2['value'] <= df_win2['global_p05']

# Per-pair (cmdb_id, kpi_name) aggregations within window
pair_agg = df_win2.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    max_value = ('value','max'),
    min_value = ('value','min'),
    total_points_window = ('value','size'),
    high_anomaly_count = ('is_high','sum'),
    low_anomaly_count = ('is_low','sum'),
    earliest_high_dt = ('dt', lambda s: s[df_win2.loc[s.index,'is_high']].min() if s.notna().any() else pd.NaT),
    earliest_low_dt  = ('dt', lambda s: s[df_win2.loc[s.index,'is_low']].min() if s.notna().any() else pd.NaT)
)

# The above lambda indexing uses s.index to refer to original df_win2 rows; ensure correct by rebuilding with apply if needed.
# Safer recompute earliest_high_dt/earliest_low_dt using explicit filtering:
eh = df_win2[df_win2['is_high']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(earliest_high_dt=('dt','min'))
el = df_win2[df_win2['is_low']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(earliest_low_dt=('dt','min'))

# Merge earliest times into pair_agg (overwrite)
pair_agg = pair_agg.drop(columns=['earliest_high_dt','earliest_low_dt'])
pair_agg = pair_agg.merge(eh, on=['cmdb_id','kpi_name'], how='left').merge(el, on=['cmdb_id','kpi_name'], how='left')

# Merge global thresholds (to ensure we exclude constant KPIs and compute deviations)
pair_agg = pair_agg.merge(df_global, on=['cmdb_id','kpi_name'], how='left')

# Filter to allowed components and exclude constant KPIs (global_p95 == global_p05)
mask_allowed = pair_agg['cmdb_id'].isin(allowed)
mask_not_constant = ~(pair_agg['global_p95'].notna() & pair_agg['global_p05'].notna() & (pair_agg['global_p95'] == pair_agg['global_p05']))
pair_agg_f = pair_agg[mask_allowed & mask_not_constant].copy()

# Compute combined anomalies
pair_agg_f['high_anomaly_count'] = pair_agg_f['high_anomaly_count'].astype(int)
pair_agg_f['low_anomaly_count']  = pair_agg_f['low_anomaly_count'].astype(int)
pair_agg_f['combined_anomaly_count'] = pair_agg_f['high_anomaly_count'] + pair_agg_f['low_anomaly_count']

# Compute relative deviations and pair severity
eps = 1e-9
pair_agg_f['high_deviation'] = np.where(
    pair_agg_f['high_anomaly_count'] > 0,
    (pair_agg_f['max_value'] - pair_agg_f['global_p95']) / np.maximum(np.abs(pair_agg_f['global_p95']), eps),
    0.0
)
pair_agg_f['low_deviation'] = np.where(
    pair_agg_f['low_anomaly_count'] > 0,
    (pair_agg_f['global_p05'] - pair_agg_f['min_value']) / np.maximum(np.abs(pair_agg_f['global_p05']), eps),
    0.0
)
pair_agg_f['pair_severity'] = np.maximum(pair_agg_f['high_deviation'], pair_agg_f['low_deviation'])

# Determine pair earliest anomaly timestamp (min of earliest_high_dt and earliest_low_dt) for pairs that have anomalies
def min_dt(a,b):
    if pd.isna(a) and pd.isna(b):
        return pd.NaT
    if pd.isna(a):
        return b
    if pd.isna(b):
        return a
    return min(a,b)

pair_agg_f['pair_earliest_anomaly_dt'] = pair_agg_f.apply(lambda r: min_dt(r['earliest_high_dt'], r['earliest_low_dt']), axis=1)

# Now aggregate per component (cmdb_id)
# Consider only pairs present in window (total_points_window>0) implicitly; but requirement counts pairs with combined_anomaly_count>0
comp_grp = pair_agg_f.groupby('cmdb_id', as_index=False).agg(
    num_faulty_kpis = ('combined_anomaly_count', lambda s: int((s>0).sum())),
    total_combined_anomalies = ('combined_anomaly_count', 'sum'),
    max_pair_severity = ('pair_severity', 'max')
)

# Compute earliest_significant_ts per component: earliest pair_earliest_anomaly_dt among pairs with pair_severity >= 0.5
sig_pairs = pair_agg_f[pair_agg_f['pair_severity'] >= 0.5].copy()
sig_earliest = sig_pairs.groupby('cmdb_id', as_index=False).agg(earliest_significant_dt = ('pair_earliest_anomaly_dt','min'))

comp_grp = comp_grp.merge(sig_earliest, on='cmdb_id', how='left')

# Convert earliest_significant_dt to ISO strings (UTC+8) or None
def to_iso_or_none(x):
    return x.isoformat() if pd.notnull(x) else None

comp_grp['earliest_significant_ts'] = comp_grp['earliest_significant_dt'].apply(lambda x: to_iso_or_none(x) if 'earliest_significant_dt' in comp_grp.columns else None)

# Select final columns
comp_result = comp_grp.loc[:, ['cmdb_id','num_faulty_kpis','total_combined_anomalies','max_pair_severity','earliest_significant_ts']]

# Count distinct candidate components after filtering (having at least one pair in pair_agg_f)
candidate_components_count = int(pair_agg_f['cmdb_id'].nunique())

# Sort by max_pair_severity desc then total_combined_anomalies desc and take top 20
df_component_agg_top20 = comp_result.sort_values(['max_pair_severity','total_combined_anomalies'], ascending=[False, False]).head(20)

# Keep output compact: show top table and candidate count
df_component_agg_top20, candidate_components_count

```
Out[4]:
```
```python
# Plain-English summary of component-level aggregation (compact), using in-memory results:
# df_component_agg_top20, candidate_components_count

lines = []
lines.append("Component analysis summary (restricted to allowed components; constant KPIs excluded):")
lines.append(f"- Total distinct candidate components after filtering: {int(candidate_components_count)}")
lines.append("- Top components (sorted by max_pair_severity desc, then total_combined_anomalies):")
for _, r in df_component_agg_top20.iterrows():
    lines.append(
        f"  - {r['cmdb_id']}: num_faulty_kpis={int(r['num_faulty_kpis'])}, "
        f"total_combined_anomalies={int(r['total_combined_anomalies'])}, "
        f"max_pair_severity={r['max_pair_severity']:.6g}, "
        f"earliest_significant_ts={r['earliest_significant_ts']}"
    )

lines.append("\nInterpretation notes:")
lines.append("- max_pair_severity measures the largest relative deviation observed for any KPI pair on the component within the incident window.")
lines.append("- earliest_significant_ts is the earliest anomaly time among pairs with severity >= 0.5 (if present).")
lines.append("- High max_pair_severity and large total_combined_anomalies point to components warranting investigation first.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  num_faulty_kpis  total_combined_anomalies  max_pair_severity    earliest_significant_ts
4    Mysql01               99                       699          19.000000  2021-03-04T01:00:00+08:00
5    Mysql02              100                       553          19.000000  2021-03-04T01:00:00+08:00
9   Tomcat02               54                       307          19.000000  2021-03-04T01:00:00+08:00
2       MG01               48                       213          14.763705  2021-03-04T01:00:00+08:00
6    Redis01               52                       404          13.746405  2021-03-04T01:00:00+08:00
8   Tomcat01               51                       271           4.126467  2021-03-04T01:00:00+08:00
13  apache02               42                       374           4.075240  2021-03-04T01:00:00+08:00
12  apache01               41                       359           2.333333  2021-03-04T01:00:00+08:00
1       IG02               53                       461           2.333333  2021-03-04T01:00:00+08:00
0       IG01               49                       319           1.742181  2021-03-04T01:01:00+08:00
11  Tomcat04               49                       261           1.656546  2021-03-04T01:00:00+08:00
10  Tomcat03               45                       194           1.186509  2021-03-04T01:00:00+08:00
7    Redis02               49                       398           1.000000  2021-03-04T01:00:00+08:00
3       MG02               50                       251           1.000000  2021-03-04T01:00:00+08:00, 14)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz

# Reuse df_trace from memory
# Ensure value numeric and dt exists
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
if 'dt' not in df_trace.columns:
    df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# 1) Global P95 and P05 per (cmdb_id, trace_name) using all data
df_trace_global = df_trace.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    global_p95 = ('value', lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan),
    global_p05 = ('value', lambda x: np.nanpercentile(x.dropna(), 5)  if x.dropna().size>0 else np.nan)
)

trace_distinct_pairs = int(df_trace_global.shape[0])

# 2) Filter trace to incident window (inclusive)
start_ts = 1614790800
end_ts   = 1614792600
df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) Merge global thresholds into windowed traces and flag anomalies
df_trace_win_thresh = df_trace_window.merge(df_trace_global, on=['cmdb_id','trace_name'], how='left')
df_trace_win_thresh['is_high'] = df_trace_win_thresh['value'] >= df_trace_win_thresh['global_p95']
df_trace_win_thresh['is_low']  = df_trace_win_thresh['value'] <= df_trace_win_thresh['global_p05']

# Aggregations per pair within window
trace_total_counts = df_trace_win_thresh.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    total_points_window=('value','size')
)

trace_high_grp = df_trace_win_thresh[df_trace_win_thresh['is_high']].groupby(['cmdb_id','trace_name'], as_index=False).agg(
    high_anomaly_count=('is_high','sum'),
    earliest_high_dt=('dt','min')
)

trace_low_grp = df_trace_win_thresh[df_trace_win_thresh['is_low']].groupby(['cmdb_id','trace_name'], as_index=False).agg(
    low_anomaly_count=('is_low','sum'),
    earliest_low_dt=('dt','min')
)

# 4) Assemble results by merging
df_trace_result = df_trace_global.merge(trace_total_counts, on=['cmdb_id','trace_name'], how='left')
df_trace_result = df_trace_result.merge(trace_high_grp, on=['cmdb_id','trace_name'], how='left')
df_trace_result = df_trace_result.merge(trace_low_grp, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for counts with 0 and format earliest timestamps
df_trace_result['total_points_window'] = df_trace_result['total_points_window'].fillna(0).astype(int)
df_trace_result['high_anomaly_count'] = df_trace_result['high_anomaly_count'].fillna(0).astype(int)
df_trace_result['low_anomaly_count']  = df_trace_result['low_anomaly_count'].fillna(0).astype(int)
df_trace_result['earliest_high_anomaly_timestamp'] = df_trace_result['earliest_high_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
df_trace_result['earliest_low_anomaly_timestamp']  = df_trace_result['earliest_low_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# 5) Restrict to allowed components and pairs with combined_anomaly_count > 0
allowed = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
           "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

df_trace_result['combined_anomaly_count'] = df_trace_result['high_anomaly_count'] + df_trace_result['low_anomaly_count']

mask_allowed = df_trace_result['cmdb_id'].isin(allowed)
mask_combined = df_trace_result['combined_anomaly_count'] > 0

# Exclude constant thresholds where global_p95 == global_p05
mask_not_constant = ~(df_trace_result['global_p95'].notna() & df_trace_result['global_p05'].notna() & (df_trace_result['global_p95'] == df_trace_result['global_p05']))

df_trace_candidates = df_trace_result[mask_allowed & mask_combined & mask_not_constant].copy()

# Sort and limit to top 20
df_trace_candidates = df_trace_candidates.sort_values(['combined_anomaly_count','total_points_window'], ascending=[False, False])
out_cols = [
    'cmdb_id','trace_name','global_p95','global_p05','total_points_window',
    'high_anomaly_count','earliest_high_anomaly_timestamp',
    'low_anomaly_count','earliest_low_anomaly_timestamp','combined_anomaly_count'
]
df_trace_result_top20 = df_trace_candidates.loc[:, out_cols].head(20)

# Store intermediate tables in memory: df_trace_global, df_trace_window, df_trace_win_thresh, trace_total_counts, trace_high_grp, trace_low_grp, df_trace_candidates
# Return compact results: top20 table and total distinct pairs examined (global)
df_trace_result_top20, trace_distinct_pairs

```
Out[5]:
```
```python
# Plain-English summary of the trace analysis (compact), using in-memory results:
# Uses variables: df_trace_result_top20, trace_distinct_pairs, df_trace_global, df_trace_window, df_trace_win_thresh, df_trace_candidates

lines = []
lines.append("Trace analysis summary (plain English):")
lines.append(f"- Total distinct (cmdb_id, trace_name) pairs examined (global): {int(trace_distinct_pairs)}")
lines.append("- Incident window: 2021-03-04 01:00:00+08 to 2021-03-04 01:30:00+08.")
lines.append("- What was computed: global P95/P05 per pair, windowed counts, high/low anomaly counts and earliest anomaly timestamps.")
lines.append(f"- Pairs reported: restricted to allowed components and those with combined_anomaly_count>0; top {min(20, len(df_trace_result_top20))} shown below.\n")
lines.append("Top trace candidates (cmdb_id | trace_name | p95 / p05 | points | high_count earliest_high | low_count earliest_low | combined):")

for _, r in df_trace_result_top20.head(20).iterrows():
    lines.append(
        f"  - {r['cmdb_id']} | {r['trace_name']} | p95={r['global_p95']:.6g} p05={r['global_p05']:.6g} | "
        f"points={int(r['total_points_window'])} | high={int(r['high_anomaly_count'])} {r['earliest_high_anomaly_timestamp']} | "
        f"low={int(r['low_anomaly_count'])} {r['earliest_low_anomaly_timestamp']} | combined={int(r['combined_anomaly_count'])}"
    )

lines.append("\nNotes:")
lines.append("- Earliest timestamps are in UTC+8.")
lines.append("- Intermediate DataFrames kept in memory for follow-up: df_trace_global, df_trace_window, df_trace_win_thresh, trace_total_counts, trace_high_grp, trace_low_grp, df_trace_candidates.")
lines.append("- If you want, I can (a) expand to all candidates, (b) aggregate by component, or (c) prioritize remediation based on combined counts or latency-related traces.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                         trace_name  global_p95  global_p05  total_points_window  high_anomaly_count earliest_high_anomaly_timestamp  low_anomaly_count earliest_low_anomaly_timestamp  combined_anomaly_count
44       MG01      trace.from_dockerA1.row_count    4.000000       1.000                   19                   3       2021-03-04T01:08:00+08:00                 10      2021-03-04T01:15:00+08:00                      13
94       MG02   trace.from_dockerB2.duration_p95    0.014440       0.007                   25                   2       2021-03-04T01:16:00+08:00                  8      2021-03-04T01:05:00+08:00                      10
80       MG02      trace.from_Tomcat03.row_count   13.000000       4.000                   30                   5       2021-03-04T01:03:00+08:00                  4      2021-03-04T01:01:00+08:00                       9
164  Tomcat03            trace.to_MG02.row_count   13.000000       4.000                   30                   5       2021-03-04T01:03:00+08:00                  4      2021-03-04T01:01:00+08:00                       9
89       MG02      trace.from_dockerA2.row_count    4.100000       1.000                   19                   1       2021-03-04T01:06:00+08:00                  8      2021-03-04T01:12:00+08:00                       9
5        IG01        trace.to_Tomcat01.row_count   14.000000       3.000                   30                   4       2021-03-04T01:03:00+08:00                  4      2021-03-04T01:20:00+08:00                       8
116  Tomcat01          trace.from_IG01.row_count   14.000000       3.000                   30                   4       2021-03-04T01:03:00+08:00                  4      2021-03-04T01:20:00+08:00                       8
20       IG02        trace.to_Tomcat01.row_count   12.000000       2.000                   30                   4       2021-03-04T01:04:00+08:00                  3      2021-03-04T01:23:00+08:00                       7
119  Tomcat01          trace.from_IG02.row_count   12.000000       2.000                   30                   4       2021-03-04T01:04:00+08:00                  3      2021-03-04T01:23:00+08:00                       7
49       MG01   trace.from_dockerB1.duration_p95    0.043840       0.006                   25                   2       2021-03-04T01:16:00+08:00                  5      2021-03-04T01:10:00+08:00                       7
104      MG02        trace.to_dockerA1.row_count   12.800000       2.000                   25                   2       2021-03-04T01:05:00+08:00                  5      2021-03-04T01:13:00+08:00                       7
11       IG01        trace.to_Tomcat03.row_count   13.000000       3.450                   30                   4       2021-03-04T01:03:00+08:00                  2      2021-03-04T01:19:00+08:00                       6
35       MG01      trace.from_Tomcat02.row_count   16.000000       4.000                   30                   3       2021-03-04T01:06:00+08:00                  3      2021-03-04T01:00:00+08:00                       6
143  Tomcat02            trace.to_MG01.row_count   16.000000       4.000                   30                   3       2021-03-04T01:06:00+08:00                  3      2021-03-04T01:00:00+08:00                       6
149  Tomcat03          trace.from_IG01.row_count   13.000000       3.450                   30                   4       2021-03-04T01:03:00+08:00                  2      2021-03-04T01:19:00+08:00                       6
51       MG01  trace.from_dockerB2.duration_mean    0.020969       0.006                   24                   2       2021-03-04T01:18:00+08:00                  4      2021-03-04T01:11:00+08:00                       6
53       MG01      trace.from_dockerB2.row_count   13.000000       1.150                   24                   4       2021-03-04T01:06:00+08:00                  2      2021-03-04T01:25:00+08:00                       6
92       MG02      trace.from_dockerB1.row_count   13.000000       2.000                   24                   3       2021-03-04T01:08:00+08:00                  3      2021-03-04T01:13:00+08:00                       6
47       MG01      trace.from_dockerA2.row_count    6.050000       1.000                   20                   1       2021-03-04T01:09:00+08:00                  5      2021-03-04T01:06:00+08:00                       6
86       MG02      trace.from_dockerA1.row_count    7.100000       1.000                   20                   1       2021-03-04T01:11:00+08:00                  5      2021-03-04T01:05:00+08:00                       6, 252)```
```

In [ ]:
# In[6]:

import re
import pandas as pd
import pytz

# Reuse df_trace_candidates from memory; if not present, raise an informative error by recomputing minimal required table
if 'df_trace_candidates' not in globals():
    # Recompute quickly from df_trace_result if available
    if 'df_trace_result' in globals():
        df_trace_candidates = df_trace_result.copy()
    else:
        raise RuntimeError("df_trace_candidates and df_trace_result not found in memory. Please run previous trace aggregation steps first.")

# Ensure allowed components list
allowed = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
           "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Pattern filters for trace_name
pattern = r'(to_Tomcat|from_Tomcat|to_MG|from_MG|to_IG|from_IG)'

# Filter rows by trace_name pattern and allowed components, and ensure combined_anomaly_count > 0 (df_trace_candidates already had this)
df_trace_focus = df_trace_candidates[
    df_trace_candidates['trace_name'].str.contains(pattern, regex=True, na=False) &
    df_trace_candidates['cmdb_id'].isin(allowed)
].copy()

# Parse direction and target from trace_name
def parse_dir_target(name):
    m = re.search(r'\b(to|from)_([A-Za-z0-9]+)', name)
    if m:
        return m.group(1), m.group(2)
    return None, None

parsed = df_trace_focus['trace_name'].apply(lambda s: pd.Series(parse_dir_target(s), index=['parsed_direction','parsed_target']))
df_trace_focus = pd.concat([df_trace_focus, parsed], axis=1)

# Convert earliest_high/low timestamp strings to timezone-aware datetimes in Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
df_trace_focus['earliest_high_dt'] = pd.to_datetime(df_trace_focus['earliest_high_anomaly_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
df_trace_focus['earliest_low_dt']  = pd.to_datetime(df_trace_focus['earliest_low_anomaly_timestamp'],  utc=True, errors='coerce').dt.tz_convert(tz)

# Compute earliest_anomaly_dt as the earlier of the two (handle NaT)
def min_dt(a,b):
    if pd.isna(a) and pd.isna(b):
        return pd.NaT
    if pd.isna(a):
        return b
    if pd.isna(b):
        return a
    return a if a <= b else b

df_trace_focus['earliest_anomaly_dt'] = df_trace_focus.apply(lambda r: min_dt(r['earliest_high_dt'], r['earliest_low_dt']), axis=1)

# Convert dt back to ISO strings (UTC+8) for output columns
df_trace_focus['earliest_high_anomaly_timestamp'] = df_trace_focus['earliest_high_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
df_trace_focus['earliest_low_anomaly_timestamp']  = df_trace_focus['earliest_low_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
df_trace_focus['earliest_anomaly_timestamp']      = df_trace_focus['earliest_anomaly_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Select required output columns
out_cols = [
    'cmdb_id', 'trace_name', 'parsed_direction', 'parsed_target',
    'combined_anomaly_count',
    'earliest_high_anomaly_timestamp', 'earliest_low_anomaly_timestamp', 'earliest_anomaly_timestamp'
]

# Sort by earliest_anomaly_timestamp asc (NaT last), then combined_anomaly_count desc
df_trace_focus_sorted = df_trace_focus.sort_values(['earliest_anomaly_dt','combined_anomaly_count'], ascending=[True, False])

# Limit to 50 rows
df_trace_focus_top50 = df_trace_focus_sorted.loc[:, out_cols].head(50)

# Keep intermediate DataFrame in memory: df_trace_focus, df_trace_focus_top50
df_trace_focus_top50

```
Out[6]:
```
```python
# Summarize the extracted trace interaction candidates (plain English) using in-memory df_trace_focus_top50
# The result will be a compact human-readable string stored in `summary`.

import pandas as pd

# Use existing dataframe from memory
df = df_trace_focus_top50.copy()

total_rows = int(df.shape[0])

# parse earliest_anomaly_timestamp column to find overall earliest (they are ISO strings or None)
if total_rows == 0:
    summary = "No trace interaction candidate rows found matching the requested patterns and filters."
else:
    # Convert to datetimes for sorting/selection (NaT for None)
    dt_series = pd.to_datetime(df['earliest_anomaly_timestamp'], utc=True, errors='coerce').dt.tz_convert('Asia/Shanghai')
    df['_earliest_dt'] = dt_series

    # overall earliest timestamp (ISO) among rows
    if df['_earliest_dt'].notna().any():
        overall_earliest = df['_earliest_dt'].min().isoformat()
    else:
        overall_earliest = None

    # Prepare top 5 examples sorted by earliest anomaly asc then combined desc
    top_examples = []
    df_sorted = df.sort_values(['_earliest_dt', 'combined_anomaly_count'], ascending=[True, False]).head(5)
    for _, r in df_sorted.iterrows():
        top_examples.append(
            f"{r['cmdb_id']} | {r['trace_name']} | direction={r['parsed_direction']} | target={r['parsed_target']} | "
            f"combined={int(r['combined_anomaly_count'])} | earliest={r['earliest_anomaly_timestamp']}"
        )

    summary_lines = [
        "Trace interaction candidate summary (filtered to allowed components & selected trace patterns):",
        f"- Total extracted rows (limited to 50): {total_rows}",
        f"- Earliest anomaly observed among these rows: {overall_earliest}",
        "- Top examples (by earliest anomaly time, up to 5):"
    ]
    summary_lines += [f"  - {e}" for e in top_examples]
    summary_lines += [
        "",
        "Notes:",
        "- Each row shows: component | trace_name | parsed direction ('to'/'from') | parsed target | combined anomaly count | earliest anomaly timestamp (UTC+8).",
        "- Many earliest anomaly timestamps are at 2021-03-04T01:00:00+08:00 (start of the incident window), indicating anomalies present from the window start for several interactions.",
        "- If you want a CSV-style table of these rows, or further aggregation by target component, I can output that next."
    ]

    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                        trace_name parsed_direction parsed_target  combined_anomaly_count earliest_high_anomaly_timestamp earliest_low_anomaly_timestamp earliest_anomaly_timestamp
35       MG01     trace.from_Tomcat02.row_count             from      Tomcat02                       6       2021-03-04T01:06:00+08:00      2021-03-04T01:00:00+08:00  2021-03-04T01:00:00+08:00
143  Tomcat02           trace.to_MG01.row_count               to          MG01                       6       2021-03-04T01:06:00+08:00      2021-03-04T01:00:00+08:00  2021-03-04T01:00:00+08:00
8        IG01       trace.to_Tomcat02.row_count               to      Tomcat02                       5       2021-03-04T01:12:00+08:00      2021-03-04T01:00:00+08:00  2021-03-04T01:00:00+08:00
131  Tomcat02         trace.from_IG01.row_count             from          IG01                       5       2021-03-04T01:12:00+08:00      2021-03-04T01:00:00+08:00  2021-03-04T01:00:00+08:00
6        IG01   trace.to_Tomcat02.duration_mean               to      Tomcat02                       4       2021-03-04T01:00:00+08:00      2021-03-04T01:21:00+08:00  2021-03-04T01:00:00+08:00
..        ...                               ...              ...           ...                     ...                             ...                            ...                        ...
125  Tomcat01           trace.to_MG01.row_count               to          MG01                       5       2021-03-04T01:04:00+08:00      2021-03-04T01:25:00+08:00  2021-03-04T01:04:00+08:00
83       MG02     trace.from_Tomcat04.row_count             from      Tomcat04                       5       2021-03-04T01:10:00+08:00      2021-03-04T01:05:00+08:00  2021-03-04T01:05:00+08:00
179  Tomcat04           trace.to_MG02.row_count               to          MG02                       5       2021-03-04T01:10:00+08:00      2021-03-04T01:05:00+08:00  2021-03-04T01:05:00+08:00
37       MG01  trace.from_Tomcat03.duration_p95             from      Tomcat03                       4       2021-03-04T01:25:00+08:00      2021-03-04T01:05:00+08:00  2021-03-04T01:05:00+08:00
160  Tomcat03        trace.to_MG01.duration_p95               to          MG01                       4       2021-03-04T01:25:00+08:00      2021-03-04T01:05:00+08:00  2021-03-04T01:05:00+08:00

[50 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd
import pytz

# Reuse in-memory df_log
# Ensure numeric and dt column
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
if 'dt' not in df_log.columns:
    df_log['dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# 1) Global P95 and P05 per (cmdb_id, log_name) using all rows
df_log_global = df_log.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    global_p95 = ('value', lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan),
    global_p05 = ('value', lambda x: np.nanpercentile(x.dropna(), 5)  if x.dropna().size>0 else np.nan)
)

# Total distinct pairs examined among allowed components will be computed later
# 2) Filter to incident window (inclusive)
start_ts = 1614790800
end_ts   = 1614792600
df_log_window = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds and compute anomaly flags
df_log_win_thresh = df_log_window.merge(df_log_global, on=['cmdb_id','log_name'], how='left')
df_log_win_thresh['is_high'] = df_log_win_thresh['value'] >= df_log_win_thresh['global_p95']
df_log_win_thresh['is_low']  = df_log_win_thresh['value'] <= df_log_win_thresh['global_p05']

# Aggregations within window
total_counts = df_log_win_thresh.groupby(['cmdb_id','log_name'], as_index=False).size().rename(columns={'size':'total_points_window'})
high_grp = df_log_win_thresh[df_log_win_thresh['is_high']].groupby(['cmdb_id','log_name'], as_index=False).agg(
    high_anomaly_count=('is_high','sum'),
    earliest_high_dt=('dt','min')
)
low_grp = df_log_win_thresh[df_log_win_thresh['is_low']].groupby(['cmdb_id','log_name'], as_index=False).agg(
    low_anomaly_count=('is_low','sum'),
    earliest_low_dt=('dt','min')
)

# 3 cont) Assemble results
df_log_result = df_log_global.merge(total_counts, on=['cmdb_id','log_name'], how='left')
df_log_result = df_log_result.merge(high_grp, on=['cmdb_id','log_name'], how='left')
df_log_result = df_log_result.merge(low_grp, on=['cmdb_id','log_name'], how='left')

# Fill counts and format timestamps
df_log_result['total_points_window'] = df_log_result['total_points_window'].fillna(0).astype(int)
df_log_result['high_anomaly_count'] = df_log_result['high_anomaly_count'].fillna(0).astype(int)
df_log_result['low_anomaly_count']  = df_log_result['low_anomaly_count'].fillna(0).astype(int)
df_log_result['combined_anomaly_count'] = df_log_result['high_anomaly_count'] + df_log_result['low_anomaly_count']

df_log_result['earliest_high_anomaly_timestamp'] = df_log_result['earliest_high_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
df_log_result['earliest_low_anomaly_timestamp']  = df_log_result['earliest_low_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# 4) Restrict to allowed components
allowed = ["Tomcat01","Tomcat02","Tomcat03","Tomcat04","apache01","apache02"]
df_log_candidates = df_log_result[df_log_result['cmdb_id'].isin(allowed)].copy()

# Total number of distinct (cmdb_id, log_name) pairs examined among allowed components
distinct_pairs_examined = int(df_log_candidates.shape[0])

# Sort by combined_anomaly_count desc and limit to top 20 for compact output
out_cols = [
    'cmdb_id','log_name','total_points_window','global_p95','global_p05',
    'high_anomaly_count','earliest_high_anomaly_timestamp',
    'low_anomaly_count','earliest_low_anomaly_timestamp','combined_anomaly_count'
]
df_log_candidates_top20 = df_log_candidates.loc[:, out_cols].sort_values(['combined_anomaly_count','total_points_window'], ascending=[False, False]).head(20)

# 4 extra) Up to 20 raw rows from filtered window where log_name == 'log.error_count' and value > 0
raw_error_rows = df_log_window[(df_log_window['log_name']=='log.error_count') & (df_log_window['value']>0)].copy().loc[:, ['timestamp','dt','cmdb_id','log_name','value']].head(20)

# Keep intermediate tables in memory: df_log_global, df_log_window, df_log_win_thresh, total_counts, high_grp, low_grp, df_log_candidates
# Return compact results (DataFrame and count)
df_log_candidates_top20, distinct_pairs_examined, raw_error_rows

```
Out[7]:
```
```python
# Plain-English summary of log analysis (compact)
summary = (
    "Log analysis summary (allowed components: Tomcat01-04, apache01-02):\n\n"
    "- Total distinct (cmdb_id, log_name) pairs examined (allowed components): 16.\n\n"
    "- log.error_count: For all Tomcat and Apache instances, global P95 and P05 are 0.0. "
    "Within the incident window each host has ~30–31 samples and every sample was flagged as BOTH high and low (combined_anomaly_count = 60–62). "
    "Earliest anomaly timestamps are 2021-03-04T01:00:00+08 (window start).\n"
    "  Note: No raw rows with log.error_count > 0 were found in the window (no actual non-zero error counts). "
    "The apparent 'anomalies' are an artifact of using >=P95 and <=P05 when P95==P05==0.\n\n"
    "- Other log KPIs: log.gc_pause_max and log.row_count show smaller, meaningful anomaly counts on several Tomcat/apache hosts "
    "(examples: Tomcat02 log.gc_pause_max combined=8; Tomcat01 log.row_count combined=5; others with combined 2–4).\n\n"
    "Key takeaway and recommended next steps:\n"
    "  1) Ignore the log.error_count dual-flagging (P95==P05==0) as it is not indicative of real errors; recompute thresholds or use strict comparisons (>) to detect true spikes.\n"
    "  2) Investigate log.gc_pause_max and log.row_count anomalies on the listed Tomcat/apache hosts (timestamps provided in the aggregated table) as they show real deviations.\n"
    "  3) If you want, I can (a) output the compact top-20 table again, (b) recompute thresholds with a different method, or (c) list the raw log rows for the non-zero gc_pause or row_count events.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  total_points_window  global_p95  global_p05  high_anomaly_count earliest_high_anomaly_timestamp  low_anomaly_count earliest_low_anomaly_timestamp  combined_anomaly_count
0   Tomcat01   log.error_count                   31      0.0000      0.0000                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
3   Tomcat02   log.error_count                   31      0.0000      0.0000                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
9   Tomcat04   log.error_count                   31      0.0000      0.0000                  31       2021-03-04T01:00:00+08:00                 31      2021-03-04T01:00:00+08:00                      62
6   Tomcat03   log.error_count                   30      0.0000      0.0000                  30       2021-03-04T01:00:00+08:00                 30      2021-03-04T01:00:00+08:00                      60
12  apache01   log.error_count                   30      0.0000      0.0000                  30       2021-03-04T01:00:00+08:00                 30      2021-03-04T01:00:00+08:00                      60
14  apache02   log.error_count                   30      0.0000      0.0000                  30       2021-03-04T01:00:00+08:00                 30      2021-03-04T01:00:00+08:00                      60
4   Tomcat02  log.gc_pause_max                   31      5.1300      5.0600                   6       2021-03-04T01:08:00+08:00                  2      2021-03-04T01:22:00+08:00                       8
2   Tomcat01     log.row_count                   31     96.0000     75.5000                   3       2021-03-04T01:08:00+08:00                  2      2021-03-04T01:29:00+08:00                       5
1   Tomcat01  log.gc_pause_max                   31      5.1350      5.0850                   2       2021-03-04T01:10:00+08:00                  2      2021-03-04T01:04:00+08:00                       4
5   Tomcat02     log.row_count                   31     96.0000     76.5000                   2       2021-03-04T01:13:00+08:00                  2      2021-03-04T01:02:00+08:00                       4
11  Tomcat04     log.row_count                   31     26.5000      7.5000                   2       2021-03-04T01:13:00+08:00                  2      2021-03-04T01:25:00+08:00                       4
8   Tomcat03     log.row_count                   30     25.1000      8.9000                   2       2021-03-04T01:07:00+08:00                  2      2021-03-04T01:18:00+08:00                       4
13  apache01     log.row_count                   30     53.5000     18.2500                   2       2021-03-04T01:13:00+08:00                  2      2021-03-04T01:26:00+08:00                       4
15  apache02     log.row_count                   30     47.6500     17.4500                   2       2021-03-04T01:09:00+08:00                  2      2021-03-04T01:20:00+08:00                       4
7   Tomcat03  log.gc_pause_max                    2      0.1245      0.0255                   1       2021-03-04T01:03:00+08:00                  1      2021-03-04T01:15:00+08:00                       2
10  Tomcat04  log.gc_pause_max                    2      0.0485      0.0215                   1       2021-03-04T01:05:00+08:00                  1      2021-03-04T01:18:00+08:00                       2, 16, Empty DataFrame
Columns: [timestamp, dt, cmdb_id, log_name, value]
Index: [])```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Ensure metric data and timezone-aware dt exist
tz = pytz.timezone('Asia/Shanghai')
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')
# Ensure value numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
# Ensure dt column (Asia/Shanghai)
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure df_global (global P95/P05 per pair) exists; recompute if missing
if 'df_global' not in globals():
    df_global = df_metric.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        global_p95 = ('value', lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan),
        global_p05 = ('value', lambda x: np.nanpercentile(x.dropna(), 5)  if x.dropna().size>0 else np.nan)
    )

# Define incident window timestamps (inclusive)
start_ts = 1614790800
end_ts   = 1614792600

# Windowed metric data (reuse df_window if exists and consistent)
if 'df_window' in globals():
    # verify it's the same window; otherwise recompute
    if not ((df_window['timestamp'] >= start_ts) & (df_window['timestamp'] <= end_ts)).all() and df_window.shape[0] > 0:
        df_win_metric = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
    else:
        df_win_metric = df_window.copy()
else:
    df_win_metric = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Merge global thresholds into windowed data
df_win_metric = df_win_metric.merge(df_global, on=['cmdb_id','kpi_name'], how='left')

# Flags for anomalies
df_win_metric['is_high'] = df_win_metric['value'] >= df_win_metric['global_p95']
df_win_metric['is_low']  = df_win_metric['value'] <= df_win_metric['global_p05']

# Target components
targets = ["Mysql01","Mysql02","Redis01","Redis02"]

# Aggregations per (cmdb_id, kpi_name) within the window
pair_stats = df_win_metric.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    total_points_window = ('value','size'),
    max_value = ('value','max'),
    min_value = ('value','min')
)

high_grp = df_win_metric[df_win_metric['is_high']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    high_anomaly_count = ('is_high','sum'),
    earliest_high_dt = ('dt','min')
)

low_grp = df_win_metric[df_win_metric['is_low']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    low_anomaly_count = ('is_low','sum'),
    earliest_low_dt = ('dt','min')
)

# Merge stats with global thresholds
pairs = pair_stats.merge(df_global, on=['cmdb_id','kpi_name'], how='left')
pairs = pairs.merge(high_grp, on=['cmdb_id','kpi_name'], how='left').merge(low_grp, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts and keep dt columns as datetimes
pairs['high_anomaly_count'] = pairs['high_anomaly_count'].fillna(0).astype(int)
pairs['low_anomaly_count']  = pairs['low_anomaly_count'].fillna(0).astype(int)
pairs['combined_anomaly_count'] = pairs['high_anomaly_count'] + pairs['low_anomaly_count']

# Compute pair_earliest_anomaly_dt as the min of earliest_high_dt and earliest_low_dt
def min_dt(a,b):
    if pd.isna(a) and pd.isna(b):
        return pd.NaT
    if pd.isna(a):
        return b
    if pd.isna(b):
        return a
    return a if a <= b else b

pairs['pair_earliest_anomaly_dt'] = pairs.apply(lambda r: min_dt(r.get('earliest_high_dt'), r.get('earliest_low_dt')), axis=1)

# Compute pair severity as defined
eps = 1e-9
pairs['high_deviation'] = np.where(
    pairs['high_anomaly_count'] > 0,
    (pairs['max_value'] - pairs['global_p95']) / np.maximum(np.abs(pairs['global_p95']), eps),
    0.0
)
pairs['low_deviation'] = np.where(
    pairs['low_anomaly_count'] > 0,
    (pairs['global_p05'] - pairs['min_value']) / np.maximum(np.abs(pairs['global_p05']), eps),
    0.0
)
pairs['pair_severity'] = np.maximum(pairs['high_deviation'], pairs['low_deviation'])

# Filter to target components and pairs with combined_anomaly_count > 0
pairs_target = pairs[pairs['cmdb_id'].isin(targets) & (pairs['combined_anomaly_count'] > 0)].copy()

# Convert earliest dt columns to ISO strings in Asia/Shanghai timezone for output
pairs_target['earliest_high_anomaly_timestamp'] = pairs_target['earliest_high_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
pairs_target['earliest_low_anomaly_timestamp']  = pairs_target['earliest_low_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
pairs_target['pair_earliest_anomaly_timestamp'] = pairs_target['pair_earliest_anomaly_dt'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Select requested columns
out_cols = [
    'cmdb_id','kpi_name','global_p95','global_p05','total_points_window',
    'max_value','min_value',
    'high_anomaly_count','earliest_high_anomaly_timestamp',
    'low_anomaly_count','earliest_low_anomaly_timestamp',
    'pair_severity','combined_anomaly_count','pair_earliest_anomaly_timestamp'
]
pairs_target = pairs_target.loc[:, out_cols]

# Sort by pair_severity desc then combined_anomaly_count desc and limit to top 20
df_top20 = pairs_target.sort_values(['pair_severity','combined_anomaly_count'], ascending=[False, False]).head(20)

# Compute earliest anomaly timestamp among these rows
if df_top20['pair_earliest_anomaly_timestamp'].notna().any():
    earliest_overall_ts = pd.to_datetime(df_top20['pair_earliest_anomaly_timestamp'], utc=True).min().tz_convert(tz).isoformat()
else:
    earliest_overall_ts = None

# Keep intermediate tables in memory: pairs, pairs_target, df_top20
df_top20, earliest_overall_ts

```
Out[8]:
```
```python
# Compact plain-English summary of top faulty KPIs for Mysql01, Mysql02, Redis01, Redis02
# Uses in-memory variables: df_top20, earliest_overall_ts

summary_lines = [
    "Metric root-cause candidate summary for Mysql01/Mysql02/Redis01/Redis02 (incident window 2021-03-04 01:00–01:30 +08):",
    "",
    f"- Earliest anomaly among these top pairs: {earliest_overall_ts}",
    "",
    "- Top observations (sorted by pair severity then anomaly counts):",
    "  * Mysql01 shows multiple very high-severity KPI spikes (pair_severity = 19, combined_anomaly_count = 20):",
    "    - Mysql-MySQL_3306_Com Insert; Innodb pages created; SlowQueries — spikes start ~2021-03-04T01:01+08 and mostly reflect increases from near-zero baselines.",
    "  * Mysql02 similarly has high-severity spikes (pair_severity = 19, combined_anomaly_count = 20):",
    "    - Innodb buffer pool pages misc; max trx lock memory bytes (example: max ~1136 vs p95 ~56.8) — earliest anomalies ~2021-03-04T01:01+08.",
    "  * Several MySQL local-disk / IO metrics show notable deviations (pair_severity ~2–3):",
    "    - sdb/sdc/sda DSKBps and DSKRTps/DSKRead on Mysql01/Mysql02 have combined anomalies and earliest anomalies at ~01:00–01:03+08.",
    "  * Redis01/Redis02 present significant issues too:",
    "    - Redis01 OSLinux CPU WIO has high relative deviation (pair_severity ≈ 13.75) with earliest anomaly ~2021-03-04T01:14+08.",
    "    - Redis01 local disk DSKBps also shows elevated deviation (pair_severity ≈ 1.15) with earliest anomaly at 01:00+08.",
    "    - Redis02 local disk DSKTps shows pair_severity = 1.0, earliest anomaly at 01:00+08.",
    "",
    "- Counts and context:",
    "  * Many top pairs have combined_anomaly_count = 20 (i.e., anomalies present in most or all window points for that KPI).",
    "  * The highest relative severities are for MySQL transactional/Innodb metrics (severity 19) and Redis CPU WIO (severity ~13.7).",
    "",
    "Interpretation and recommended next steps:",
    "  1) Prioritize investigation of Mysql01 and Mysql02 Innodb / transaction metrics (Com Insert, Innodb pages created, SlowQueries, trx lock memory) — these show the largest relative spikes beginning ~01:01+08.",
    "  2) Investigate Redis01 CPU WIO and Redis disk activity next (high relative deviations and early timestamps).",
    "  3) Check underlying disk I/O on Mysql hosts (sda/sdb/sdc) since multiple DSK* metrics deviate concurrently.",
    "",
    "If you want, I can (a) expand this to the full list of affected KPIs, (b) provide the raw windowed samples for a chosen KPI, or (c) correlate these metric spikes with trace interactions or logs."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  global_p95  global_p05  total_points_window  max_value  min_value  high_anomaly_count earliest_high_anomaly_timestamp  low_anomaly_count earliest_low_anomaly_timestamp  pair_severity  combined_anomaly_count pair_earliest_anomaly_timestamp
373  Mysql01                        Mysql-MySQL_3306_Com Insert    0.020000    0.000000                   20     0.4000   0.000000                   1       2021-03-04T01:01:00+08:00                 19      2021-03-04T01:02:00+08:00      19.000000                      20       2021-03-04T01:01:00+08:00
433  Mysql01              Mysql-MySQL_3306_Innodb pages created    0.000835    0.000000                   20     0.0167   0.000000                   1       2021-03-04T01:01:00+08:00                 19      2021-03-04T01:02:00+08:00      19.000000                      20       2021-03-04T01:01:00+08:00
470  Mysql01                       Mysql-MySQL_3306_SlowQueries    0.000835    0.000000                   20     0.0167   0.000000                   1       2021-03-04T01:14:00+08:00                 19      2021-03-04T01:01:00+08:00      19.000000                      20       2021-03-04T01:01:00+08:00
628  Mysql02     Mysql-MySQL_3306_Innodb buffer pool pages misc    0.000835    0.000000                   20     0.0167   0.000000                   1       2021-03-04T01:16:00+08:00                 19      2021-03-04T01:01:00+08:00      19.000000                      20       2021-03-04T01:01:00+08:00
708  Mysql02         Mysql-MySQL_3306_max trx lock memory bytes   56.800000    0.000000                   20  1136.0000   0.000000                   1       2021-03-04T01:07:00+08:00                 19      2021-03-04T01:01:00+08:00      19.000000                      20       2021-03-04T01:01:00+08:00
815  Redis01                             OSLinux-CPU_CPU_CPUWio    0.016621    0.004155                   31     0.2451   0.004155                   2       2021-03-04T01:14:00+08:00                  3      2021-03-04T01:24:00+08:00      13.746405                       5       2021-03-04T01:14:00+08:00
426  Mysql01         Mysql-MySQL_3306_Innodb log write requests    0.200030    0.032470                   20     0.8333   0.016700                   1       2021-03-04T01:01:00+08:00                  1      2021-03-04T01:22:00+08:00       3.165875                       2       2021-03-04T01:01:00+08:00
549  Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.250000    0.000000                   16     1.0000   0.000000                   1       2021-03-04T01:10:00+08:00                 15      2021-03-04T01:00:00+08:00       3.000000                      16       2021-03-04T01:00:00+08:00
366  Mysql01                  Mysql-MySQL_3306_Binlog Cache Use    0.209135    0.033300                   20     0.7000   0.033300                   1       2021-03-04T01:01:00+08:00                  5      2021-03-04T01:22:00+08:00       2.347120                       6       2021-03-04T01:01:00+08:00
551  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.005010    0.000000                   15     0.0167   0.000000                   1       2021-03-04T01:18:00+08:00                 14      2021-03-04T01:01:00+08:00       2.333333                      15       2021-03-04T01:01:00+08:00
552  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.039990    0.000000                   15     0.1333   0.000000                   1       2021-03-04T01:18:00+08:00                 14      2021-03-04T01:01:00+08:00       2.333333                      15       2021-03-04T01:01:00+08:00
560  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRTps    0.005010    0.000000                   15     0.0167   0.000000                   1       2021-03-04T01:01:00+08:00                 14      2021-03-04T01:03:00+08:00       2.333333                      15       2021-03-04T01:01:00+08:00
561  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRead    0.039990    0.000000                   15     0.1333   0.000000                   1       2021-03-04T01:01:00+08:00                 14      2021-03-04T01:03:00+08:00       2.333333                      15       2021-03-04T01:01:00+08:00
762  Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.300000    0.000000                   15     1.0000   0.000000                   1       2021-03-04T01:11:00+08:00                 14      2021-03-04T01:00:00+08:00       2.333333                      15       2021-03-04T01:00:00+08:00
771  Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    1.200000    0.000000                   15     4.0000   0.000000                   1       2021-03-04T01:03:00+08:00                 14      2021-03-04T01:01:00+08:00       2.333333                      15       2021-03-04T01:01:00+08:00
414  Mysql01  Mysql-MySQL_3306_Innodb buffer pool write requ...    3.200030    0.900000                   20     8.5833   0.900000                   1       2021-03-04T01:01:00+08:00                  4      2021-03-04T01:22:00+08:00       1.682256                       5       2021-03-04T01:01:00+08:00
432  Mysql01             Mysql-MySQL_3306_Innodb os log written  521.813335   67.840030                   20  1194.6667  59.733300                   1       2021-03-04T01:01:00+08:00                  1      2021-03-04T01:22:00+08:00       1.289452                       2       2021-03-04T01:01:00+08:00
781  Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKPer...   16.680000    0.560000                   15    36.0000   0.000000                   1       2021-03-04T01:27:00+08:00                  1      2021-03-04T01:23:00+08:00       1.158273                       2       2021-03-04T01:23:00+08:00
844  Redis01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    3.250000    0.000000                   16     7.0000   0.000000                   1       2021-03-04T01:04:00+08:00                 11      2021-03-04T01:00:00+08:00       1.153846                      12       2021-03-04T01:00:00+08:00
966  Redis02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    0.500000    0.000000                   11     1.0000   0.000000                   1       2021-03-04T01:06:00+08:00                 10      2021-03-04T01:00:00+08:00       1.000000                      11       2021-03-04T01:00:00+08:00, '2021-03-04T01:00:00+08:00')```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing in-memory dfs where possible
tz = pytz.timezone('Asia/Shanghai')

# Ensure df_metric is available
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')
# Ensure numeric and dt column
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure df_global (global P95/P05 per pair) exists; compute if missing
if 'df_global' not in globals():
    df_global = df_metric.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        global_p95 = ('value', lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan),
        global_p05 = ('value', lambda x: np.nanpercentile(x.dropna(), 5)  if x.dropna().size>0 else np.nan)
    )

# Incident window
start_ts = 1614790800
end_ts   = 1614792600

# Windowed metric data (reuse df_window if present)
if 'df_window' in globals():
    df_win_metric = df_window.copy()
else:
    df_win_metric = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Merge global thresholds into windowed data
df_win_metric = df_win_metric.merge(df_global, on=['cmdb_id','kpi_name'], how='left')

# Flags for anomalies using global thresholds
df_win_metric['is_high'] = df_win_metric['value'] >= df_win_metric['global_p95']
df_win_metric['is_low']  = df_win_metric['value'] <= df_win_metric['global_p05']

# Compute earliest timestamps (as integer epoch seconds) for highs and lows per pair within window
high_ts = df_win_metric[df_win_metric['is_high']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(earliest_high_ts=('timestamp','min'))
low_ts  = df_win_metric[df_win_metric['is_low']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(earliest_low_ts =('timestamp','min'))

# Merge earliest ts with global thresholds for the requested pairs
targets = {
    "Mysql01": [
        "Mysql-MySQL_3306_Com Insert",
        "Mysql-MySQL_3306_Innodb pages created",
        "Mysql-MySQL_3306_SlowQueries",
        "Mysql-MySQL_3306_Innodb log write requests",
        "Mysql-MySQL_3306_Innodb os log written"
    ],
    "Mysql02": [
        "Mysql-MySQL_3306_Innodb buffer pool pages misc",
        "Mysql-MySQL_3306_max trx lock memory bytes"
    ]
}

pairs_list = []
for cmdb, kpis in targets.items():
    for k in kpis:
        pairs_list.append((cmdb, k))

df_pairs = pd.DataFrame(pairs_list, columns=['cmdb_id','kpi_name'])

# Attach global thresholds
df_pairs = df_pairs.merge(df_global, on=['cmdb_id','kpi_name'], how='left')

# Attach earliest anomaly timestamps (ints) if any
df_pairs = df_pairs.merge(high_ts, on=['cmdb_id','kpi_name'], how='left').merge(low_ts, on=['cmdb_id','kpi_name'], how='left')

# Compute earliest_anomaly_ts_int as min of available earliest_high_ts and earliest_low_ts
def min_ts(a,b):
    if pd.isna(a) and pd.isna(b):
        return np.nan
    if pd.isna(a):
        return b
    if pd.isna(b):
        return a
    return int(min(a,b))

df_pairs['earliest_anomaly_ts'] = df_pairs.apply(lambda r: min_ts(r.get('earliest_high_ts'), r.get('earliest_low_ts')), axis=1)

# For each pair, fetch value at earliest_anomaly_ts and previous timestamp's value (one minute earlier)
def fetch_values(row):
    ts = row['earliest_anomaly_ts']
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    if pd.isna(ts):
        return pd.Series([None, None, None, None])
    ts = int(ts)
    # value at timestamp
    mask_cur = (df_win_metric['cmdb_id']==cmdb) & (df_win_metric['kpi_name']==kpi) & (df_win_metric['timestamp']==ts)
    cur_rows = df_win_metric.loc[mask_cur]
    value_at = float(cur_rows['value'].iat[0]) if not cur_rows.empty else None
    # previous timestamp
    prev_ts = ts - 60
    mask_prev = (df_win_metric['cmdb_id']==cmdb) & (df_win_metric['kpi_name']==kpi) & (df_win_metric['timestamp']==prev_ts)
    prev_rows = df_win_metric.loc[mask_prev]
    prev_val = float(prev_rows['value'].iat[0]) if not prev_rows.empty else None
    return pd.Series([pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz).isoformat(), value_at, prev_ts if prev_rows.shape[0]>0 else None, prev_val])

df_pairs[['earliest_anomaly_timestamp','value_at_earliest','previous_timestamp','previous_value']] = df_pairs.apply(fetch_values, axis=1)

# Keep only requested output columns and limit to the listed pairs order
out_cols = ['cmdb_id','kpi_name','global_p95','global_p05','earliest_anomaly_timestamp','value_at_earliest','previous_timestamp','previous_value']
df_confirm = df_pairs.loc[:, out_cols]

# Preserve original requested order
df_confirm['__order'] = df_confirm.apply(lambda r: pairs_list.index((r['cmdb_id'], r['kpi_name'])), axis=1)
df_confirm = df_confirm.sort_values('__order').drop(columns='__order').reset_index(drop=True)

# Display compact result (at most 7 rows as requested)
df_confirm

```
Out[9]:
```
Summary of the 7 requested KPI pairs (incident window 2021-03-04 01:00–01:30 +08):

All pairs have their earliest detected anomaly at 2021-03-04T01:01:00+08. Previous-minute samples (one minute earlier) were not present in the window for these pairs (previous_timestamp = None), so we can only compare against the earliest observed value at 01:01.

Per-pair details:
- Mysql01 | Mysql-MySQL_3306_Com Insert
  - global_p95 = 0.02, global_p05 = 0.0
  - earliest_anomaly = 2021-03-04T01:01:00+08
  - value_at_earliest = 0.4000 → this is a clear high spike (0.4 >= p95)

- Mysql01 | Mysql-MySQL_3306_Innodb pages created
  - global_p95 = 0.000835, global_p05 = 0.0
  - earliest_anomaly = 2021-03-04T01:01:00+08
  - value_at_earliest = 0.0167 → clear high spike (0.0167 >> p95)

- Mysql01 | Mysql-MySQL_3306_SlowQueries
  - global_p95 = 0.000835, global_p05 = 0.0
  - earliest_anomaly = 2021-03-04T01:01:00+08
  - value_at_earliest = 0.0000 → this earliest anomaly is a low/reference (value == 0, not a high spike)

- Mysql01 | Mysql-MySQL_3306_Innodb log write requests
  - global_p95 = 0.20003, global_p05 = 0.03247
  - earliest_anomaly = 2021-03-04T01:01:00+08
  - value_at_earliest = 0.8333 → clear high spike (0.8333 >= p95)

- Mysql01 | Mysql-MySQL_3306_Innodb os log written
  - global_p95 = 521.813335, global_p05 = 67.84003
  - earliest_anomaly = 2021-03-04T01:01:00+08
  - value_at_earliest = 1194.6667 → high spike (≈2.29× p95)

- Mysql02 | Mysql-MySQL_3306_Innodb buffer pool pages misc
  - global_p95 = 0.000835, global_p05 = 0.0
  - earliest_anomaly = 2021-03-04T01:01:00+08
  - value_at_earliest = 0.0000 → earliest anomaly is a low/reference (value == 0) — no high spike at this earliest timestamp

- Mysql02 | Mysql-MySQL_3306_max trx lock memory bytes
  - global_p95 = 56.8, global_p05 = 0.0
  - earliest_anomaly = 2021-03-04T01:01:00+08
  - value_at_earliest = 0.0000 → earliest anomaly is a low/reference (value == 0). (Note: a large high value for this KPI was observed later in the window in prior analysis — but the earliest anomaly record here corresponds to a low value at 01:01.)

Conclusions / guidance:
- Several MySQL metrics (Com Insert, Innodb pages created, Innodb log write requests, os log written) show clear high spikes at 01:01+08 and are strong candidates for root cause investigation starting at that time.
- For pairs whose earliest anomaly is a low (value 0 at 01:01), the actual harmful high spike may occur later in the window; check the full time series for those KPIs (Mysql02 max trx lock memory bytes, buffer pool pages misc) to find the spike time and magnitude.
- Because there is no previous-minute sample inside the window for these earliest timestamps, confirm behavior just before 01:01 by checking raw series including 01:00 if available outside the current window.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                        kpi_name  global_p95  global_p05 earliest_anomaly_timestamp  value_at_earliest previous_timestamp previous_value
0  Mysql01                     Mysql-MySQL_3306_Com Insert    0.020000     0.00000  2021-03-04T01:01:00+08:00             0.4000               None           None
1  Mysql01           Mysql-MySQL_3306_Innodb pages created    0.000835     0.00000  2021-03-04T01:01:00+08:00             0.0167               None           None
2  Mysql01                    Mysql-MySQL_3306_SlowQueries    0.000835     0.00000  2021-03-04T01:01:00+08:00             0.0000               None           None
3  Mysql01      Mysql-MySQL_3306_Innodb log write requests    0.200030     0.03247  2021-03-04T01:01:00+08:00             0.8333               None           None
4  Mysql01          Mysql-MySQL_3306_Innodb os log written  521.813335    67.84003  2021-03-04T01:01:00+08:00          1194.6667               None           None
5  Mysql02  Mysql-MySQL_3306_Innodb buffer pool pages misc    0.000835     0.00000  2021-03-04T01:01:00+08:00             0.0000               None           None
6  Mysql02      Mysql-MySQL_3306_max trx lock memory bytes   56.800000     0.00000  2021-03-04T01:01:00+08:00             0.0000               None           None```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# Reuse existing df_metric in memory
tz = pytz.timezone('Asia/Shanghai')

# Ensure df_metric exists
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')

# Ensure numeric value and dt column with UTC+8 timezone
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure dt is timezone-aware in Asia/Shanghai
    if pd.api.types.is_datetime64_ns_dtype(df_metric['dt'].dt.tz) if hasattr(df_metric['dt'], 'dt') else False:
        df_metric['dt'] = pd.to_datetime(df_metric['dt'], errors='coerce').dt.tz_convert(tz)
    else:
        # recreate from timestamp to be safe
        df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define target component and KPI list
component = "Mysql01"
kpi_list = [
    "Mysql-MySQL_3306_Com Insert",
    "Mysql-MySQL_3306_Innodb pages created",
    "Mysql-MySQL_3306_SlowQueries",
    "Mysql-MySQL_3306_Innodb log write requests",
    "Mysql-MySQL_3306_Innodb os log written"
]

# Incident sub-window: 2021-03-04 01:00:00+08 to 2021-03-04 01:05:00+08 inclusive
start_ts = 1614790800
end_ts   = 1614791100

# Filter rows for Mysql01, selected KPIs, and timestamp window
df_mysql_window = df_metric[
    (df_metric['cmdb_id'] == component) &
    (df_metric['kpi_name'].isin(kpi_list)) &
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts)
].copy()

# Sort by kpi_name then timestamp ascending and take up to 10 rows per KPI
df_mysql_window_sorted = df_mysql_window.sort_values(['kpi_name','timestamp'], ascending=[True, True])
df_mysql_rows = df_mysql_window_sorted.groupby('kpi_name', group_keys=False).head(10)

# Select requested columns and reset index for compact display
df_mysql_rows = df_mysql_rows.loc[:, ['timestamp','dt','cmdb_id','kpi_name','value']].reset_index(drop=True)

# Display the compact table
df_mysql_rows

```
Out[10]:
```
```python
# Summarize exact minute and abruptness for selected Mysql01 KPIs using df_mysql_rows (already in memory)
import pandas as pd

# Use df_mysql_rows computed earlier
df = df_mysql_rows.copy()

summary_lines = []
summary_lines.append("Mysql01 KPI spike confirmation (window 2021-03-04 01:00–01:05 +08):")
summary_lines.append("")

# For each KPI, find earliest observed row and next row (if any) within the extracted window
for kpi, grp in df.groupby('kpi_name'):
    grp = grp.sort_values('timestamp').reset_index(drop=True)
    first_ts = grp.loc[0, 'dt'].isoformat()
    first_val = grp.loc[0, 'value']
    # find next row after earliest (if exists)
    next_val = None
    next_ts = None
    if grp.shape[0] > 1:
        next_ts = grp.loc[1, 'dt'].isoformat()
        next_val = grp.loc[1, 'value']
    # Determine abruptness
    if next_val is None:
        abrupt = "no later sample in window to compare"
    else:
        if pd.isna(first_val) or pd.isna(next_val):
            abrupt = "missing value(s) to assess abruptness"
        else:
            # treat as abrupt if change magnitude is large relative to first_val baseline or drops to zero
            if first_val == 0:
                if next_val == 0:
                    abrupt = "no spike (stays zero)"
                else:
                    abrupt = "increase after earliest sample"
            else:
                rel_change = (first_val - next_val) / (abs(first_val) + 1e-9)
                if rel_change > 0.5 or next_val == 0:
                    abrupt = "abrupt spike at earliest timestamp (value drops markedly next minute)"
                else:
                    abrupt = "elevated at earliest timestamp but not abruptly different next minute"

    summary_lines.append(f"- {kpi}: earliest={first_ts}, value_at_earliest={first_val}; "
                         + (f"next={next_ts} value={next_val}; {abrupt}" if next_val is not None else abrupt))

# Overall conclusion
summary_lines.append("")
summary_lines.append("Conclusion:")
summary_lines.append("- Multiple Mysql01 KPIs (Com Insert, Innodb pages created, Innodb log write requests, os log written) show clear spikes at 2021-03-04T01:01:00+08.")
summary_lines.append("- For these KPIs the value at 01:01 is substantially higher than subsequent minutes (often dropping the next minute), indicating an abrupt spike starting at 01:01+08.")
summary_lines.append("- SlowQueries remained zero in the examined window (no spike at 01:01).")
summary_lines.append("- Final root-cause occurrence time for Mysql01 metrics: 2021-03-04T01:01:00+08 (spike begins at this minute).")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                        dt  cmdb_id                                    kpi_name      value
0   1614790860 2021-03-04 01:01:00+08:00  Mysql01                 Mysql-MySQL_3306_Com Insert     0.4000
1   1614790920 2021-03-04 01:02:00+08:00  Mysql01                 Mysql-MySQL_3306_Com Insert     0.0000
2   1614791040 2021-03-04 01:04:00+08:00  Mysql01                 Mysql-MySQL_3306_Com Insert     0.0000
3   1614791100 2021-03-04 01:05:00+08:00  Mysql01                 Mysql-MySQL_3306_Com Insert     0.0000
4   1614790860 2021-03-04 01:01:00+08:00  Mysql01  Mysql-MySQL_3306_Innodb log write requests     0.8333
5   1614790920 2021-03-04 01:02:00+08:00  Mysql01  Mysql-MySQL_3306_Innodb log write requests     0.1667
6   1614791040 2021-03-04 01:04:00+08:00  Mysql01  Mysql-MySQL_3306_Innodb log write requests     0.0667
7   1614791100 2021-03-04 01:05:00+08:00  Mysql01  Mysql-MySQL_3306_Innodb log write requests     0.1000
8   1614790860 2021-03-04 01:01:00+08:00  Mysql01      Mysql-MySQL_3306_Innodb os log written  1194.6667
9   1614790920 2021-03-04 01:02:00+08:00  Mysql01      Mysql-MySQL_3306_Innodb os log written   384.0000
10  1614791040 2021-03-04 01:04:00+08:00  Mysql01      Mysql-MySQL_3306_Innodb os log written   153.6000
11  1614791100 2021-03-04 01:05:00+08:00  Mysql01      Mysql-MySQL_3306_Innodb os log written   213.3333
12  1614790860 2021-03-04 01:01:00+08:00  Mysql01       Mysql-MySQL_3306_Innodb pages created     0.0167
13  1614790920 2021-03-04 01:02:00+08:00  Mysql01       Mysql-MySQL_3306_Innodb pages created     0.0000
14  1614791040 2021-03-04 01:04:00+08:00  Mysql01       Mysql-MySQL_3306_Innodb pages created     0.0000
15  1614791100 2021-03-04 01:05:00+08:00  Mysql01       Mysql-MySQL_3306_Innodb pages created     0.0000
16  1614790860 2021-03-04 01:01:00+08:00  Mysql01                Mysql-MySQL_3306_SlowQueries     0.0000
17  1614790920 2021-03-04 01:02:00+08:00  Mysql01                Mysql-MySQL_3306_SlowQueries     0.0000
18  1614791040 2021-03-04 01:04:00+08:00  Mysql01                Mysql-MySQL_3306_SlowQueries     0.0000
19  1614791100 2021-03-04 01:05:00+08:00  Mysql01                Mysql-MySQL_3306_SlowQueries     0.0000```
```